# Fase 1 — Entendimento do Negócio

**Projeto:** Previsão de cancelamento (churn) de clientes de uma operadora de telecomunicações
**Metodologia:** CRISP-DM
**Disciplina:** Ciência de Dados e Aprendizagem de Máquina — Profa. Kadidja Valéria

O objetivo desta fase é traduzir o problema de negócio em metas técnicas viáveis **antes** de investir esforço em dados e modelos. Ela segue o checklist do guia:

1. Determinar os objetivos de negócio e os critérios de sucesso
2. Avaliar a situação (recursos, riscos, custo-benefício)
3. Determinar as metas técnicas de mineração de dados
4. Produzir o plano de projeto

## 1. Objetivos de negócio e critérios de sucesso

### Cenário

Uma operadora de telecomunicações (telefonia, internet e streaming) está perdendo clientes. Conquistar um cliente novo custa mais do que manter um cliente atual, e hoje a empresa só descobre que o cliente vai sair quando ele pede o cancelamento, momento em que já é tarde para agir.

> O cenário é uma simulação didática. Os dados são o conjunto público *Telco Customer Churn* (amostra da IBM, publicada no Kaggle), tratado aqui como se fosse a base de clientes da operadora.

A célula abaixo mede o tamanho do problema na base disponível.

In [1]:
from pathlib import Path
import pandas as pd

RAIZ = Path("..").resolve()
df = pd.read_csv(RAIZ / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv")

total_clientes = len(df)
cancelaram = (df["Churn"] == "Yes").sum()
receita_mensal = df["MonthlyCharges"].sum()
receita_perdida = df.loc[df["Churn"] == "Yes", "MonthlyCharges"].sum()

print(f"Clientes na base:                 {total_clientes:,}")
print(f"Clientes que cancelaram:          {cancelaram:,} ({cancelaram / total_clientes:.1%})")
print(f"Receita mensal da base:           $ {receita_mensal:,.2f}")
print(f"Receita mensal dos que cancelaram: $ {receita_perdida:,.2f} ({receita_perdida / receita_mensal:.1%})")
print(f"Mensalidade média de quem cancelou: $ {df.loc[df['Churn'] == 'Yes', 'MonthlyCharges'].mean():.2f}")
print(f"Mensalidade média de quem ficou:    $ {df.loc[df['Churn'] == 'No', 'MonthlyCharges'].mean():.2f}")

Clientes na base:                 7,043
Clientes que cancelaram:          1,869 (26.5%)
Receita mensal da base:           $ 456,116.60
Receita mensal dos que cancelaram: $ 139,130.85 (30.5%)
Mensalidade média de quem cancelou: $ 74.44
Mensalidade média de quem ficou:    $ 61.27


Cerca de um em cada quatro clientes cancelou (26,5%), e eles representam uma fatia ainda maior da receita (30,5%), porque quem cancela paga, em média, uma mensalidade mais alta do que quem fica. O problema é relevante o suficiente para justificar o projeto.

### Objetivo de negócio

**Reduzir a perda de receita por cancelamento, identificando com antecedência os clientes com maior risco de sair para que a equipe de retenção faça uma oferta direcionada a eles.**

### Critérios de sucesso do negócio

Definidos agora, antes de qualquer modelo, e usados como régua na Fase 5 (Avaliação):

| # | Critério | Como será medido |
|---|---|---|
| CN1 | A campanha deve alcançar pelo menos **70% dos clientes que cancelariam** | Proporção dos cancelamentos reais que o modelo marca para contato |
| CN2 | A campanha direcionada deve dar **retorno líquido positivo e maior do que contatar toda a base** | Simulação financeira com as premissas da seção 2 |
| CN3 | Os fatores de risco apontados devem **fazer sentido comercial** | Análise dos fatores mais importantes do modelo |

## 2. Avaliação da situação

### Recursos

| Recurso | Disponível |
|---|---|
| Dados | Uma tabela com 7.043 clientes e 21 atributos (cadastro, serviços contratados, cobrança e indicação de cancelamento) |
| Ferramentas | Python, pandas, scikit-learn, matplotlib/seaborn, Jupyter |
| Pessoas | Um analista (o autor), acumulando os papéis de negócio e técnico |

### Riscos e contingências

| Risco | Impacto | Contingência |
|---|---|---|
| A base é uma fotografia de um único momento, sem datas | Não é possível validar o modelo ao longo do tempo | Separar um conjunto de teste e prever monitoramento na Fase 6 |
| Não há dados de custo de retenção nem de sucesso de ofertas | O retorno financeiro não pode ser medido, só simulado | Declarar as premissas e testar a sensibilidade do resultado a elas |
| Classes desbalanceadas (cerca de 27% de cancelamentos) | A acurácia sozinha engana | Usar ROC-AUC, recall e precisão, e ajustar o limiar de decisão |
| Atributos sensíveis (gênero, idade) | Tratamento desigual de clientes | Verificar na avaliação o peso desses atributos no modelo |

### Custo-benefício: premissas

O dataset não traz custos nem resultados de campanhas. Para avaliar o impacto financeiro, o projeto adota **premissas**, que ficam registradas em `src/negocio.py` e devem ser trocadas por valores reais quando existirem:

| Premissa | Valor adotado |
|---|---|
| Custo da oferta de retenção por cliente contatado | $ 50 |
| Clientes em risco que aceitam a oferta e permanecem | 30% |
| Receita preservada por cliente retido | 12 mensalidades do próprio cliente |

Com essas premissas, errar para os dois lados tem custos diferentes: contatar um cliente que não ia sair custa $ 50, enquanto deixar de contatar um cliente que sai custa, em média, 0,30 × 12 × $ 74 ≈ $ 268 de receita não preservada. Isso indica que o modelo deve priorizar **encontrar quem vai cancelar** (recall), mesmo que isso gere alguns contatos desnecessários.

## 3. Metas técnicas de mineração de dados

| Meta de negócio | Meta técnica |
|---|---|
| Identificar clientes em risco de cancelar | Modelo de **classificação binária** que estima a probabilidade de churn de cada cliente |
| Direcionar a campanha | Definir um **limiar de probabilidade** acima do qual o cliente entra na campanha |
| Entender os motivos | Identificar os atributos que mais influenciam a previsão |

### Critérios de sucesso técnicos

| # | Critério |
|---|---|
| CT1 | ROC-AUC ≥ 0,80 no conjunto de teste |
| CT2 | Recall da classe "cancelou" ≥ 0,70 no conjunto de teste, no limiar escolhido |
| CT3 | Desempenho no teste próximo ao da validação cruzada (sem sinal de sobreajuste) |

## 4. Plano de projeto

| Fase | Atividades principais | Entregável |
|---|---|---|
| 1. Entendimento do Negócio | Objetivos, critérios de sucesso, premissas, plano | Este notebook |
| 2. Entendimento dos Dados | Coleta, descrição, exploração, verificação de qualidade | `02_entendimento_dos_dados.ipynb` |
| 3. Preparação dos Dados | Selecionar, limpar, construir, integrar | `03_preparacao_dos_dados.ipynb`, `src/preparacao.py`, base preparada |
| 4. Modelagem | Seleção de técnicas, desenho de teste, construção e ajuste | `04_modelagem.ipynb`, modelo salvo |
| 5. Avaliação | Conferência com os critérios desta fase, revisão do processo, decisão | `05_avaliacao.ipynb` |
| 6. Implementação | Plano de implementação e monitoramento, relatório final, lições aprendidas | `06_implementacao.ipynb`, lista de retenção de exemplo |

O ciclo não é rígido: as fases 2 a 5 podem devolver o projeto a uma fase anterior. Os retornos que aconteceram neste projeto estão registrados no notebook da Fase 6.